In [1]:
model = "llama3.2:3b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0,
)

In [3]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434",
)

In [4]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_collection(name="ai_model_book")

# Create chromadb
vector_db_from_client = Chroma(
    client=client,
    collection_name="ai_model_book",
    embedding_function=embedding_model,
)

print("Gespeicherte Textabschnitte:", collection.count())

Gespeicherte Textabschnitte: 2680


In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the main themes in at most three sentences using only the retrieved documents. "
     "Preserve the distinctions between categories in the source; do not invent relationships or examples."),
    ("human", "Retrieved documents:\n{docs}\n\nSummarize their main themes."),
])


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = (
    {"docs": format_docs}
    | prompt
    | model
    | StrOutputParser()
)

In [6]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(
    search_query,
    k=3,
)

In [7]:
print(chain.invoke(docs))

Here is a summary of the main themes in three sentences:

Machine Learning systems can be classified based on the type of supervision they receive during training, including supervised, unsupervised, semisupervised, and Reinforcement Learning. Supervised learning involves training with labeled data, where the algorithm learns from examples with desired solutions, whereas unsupervised learning involves discovering patterns in data without prior labels. These categories can be further combined with other criteria, such as online versus batch learning and instance-based versus model-based learning.


In [8]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)
print()

Here is a summary of the main themes in three sentences:

Machine Learning systems can be classified based on the type of supervision they receive during training, including supervised, unsupervised, semisupervised, and Reinforcement Learning. Supervised learning involves training with labeled data, while unsupervised learning involves training without labeled data. The classification of Machine Learning systems can also be based on whether they learn incrementally on the fly or build a predictive model from existing data.


In [9]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)
print()

Here is a summary of the main themes in three sentences:

Machine Learning systems can be classified based on the type of supervision they receive during training, including supervised, unsupervised, semisupervised, and Reinforcement Learning. Supervised learning involves training with labeled data, while unsupervised learning involves training without labeled data. The classification of Machine Learning systems can also be based on whether they learn incrementally on the fly or build a predictive model from existing data.


#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [10]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

rag_prompt = ChatPromptTemplate.from_template(prompt_template)

retriever = vector_db_from_client.as_retriever(
    search_kwargs={"k": 3}
)

qa_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

In [11]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x112e5ab90>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6'}}, model='llama3.2:3b', temperature=0.0, base_url='http://localhos

In [12]:
question = "What is supervised learning?"

print(qa_rag_chain.invoke(question))

Supervised learning is a type of machine learning where the training data includes the desired solutions, called labels, to guide the algorithm in making predictions. In supervised learning, the algorithm is trained on labeled data to learn the relationship between the input features and the output labels. This type of learning is "based" and relies on the supervision provided by the labeled data.


In [13]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the training data includes the desired solutions, called labels, to guide the algorithm in making predictions. In supervised learning, the algorithm learns from labeled data to make accurate predictions. The training data is used to teach the algorithm what the correct output should be for a given input.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [14]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [15]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

/var/folders/pd/c2__f1vx5fxfx3163pqvn8k80000gn/T/ipykernel_82372/2064999211.py:2: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
/Users/berkan/build-your-own-chatbot-stud/.venv/lib/python3.11/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validat

In [16]:
conversation_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only sentences from the retrieved context that directly answer the question. "
     "Do not combine statements about different concepts. Keep the answer to at most three sentences. "
     "If the context does not support an answer, say you do not know."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

qa_chain = ConversationalRetrievalChain.from_llm(
    model,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": conversation_prompt},
    verbose=False,
)

In [17]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

In supervised learning, the training data you feed to the algorithm includes the desired solutions, called labels.

In [18]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Can supervised learning be implemented using various algorithms?Yes, supervised learning can be implemented using various algorithms, such as k-Nearest Neighbors, Linear Regression, Logistic Regression, Support Vector Machines (SVMs), Decision Trees and Random Forests, and Neural networks.